In [1]:
"""
A3T-GCN Multi-Horizonte - REMMAQ
Ejecutar una vez por horizonte cambiando HORIZON y SEQ_LEN abajo.

Ejecuciones planificadas:
  Corrida 1: HORIZON = 6,  SEQ_LEN = 24  (~8 min/epoch)
  Corrida 2: HORIZON = 12, SEQ_LEN = 48  (~16 min/epoch)
  Corrida 3: HORIZON = 24, SEQ_LEN = 48  (~16 min/epoch)
"""

import numpy as np
import pandas as pd
import pickle
import time
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from torch_geometric_temporal.nn.recurrent import A3TGCN

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# >>> CAMBIAR ESTOS DOS VALORES PARA CADA CORRIDA <<<
# ============================================================
HORIZON = 6       # <-- Cambiar a 6, 12 o 24
SEQ_LEN = 24      # <-- 24 para horizon 6; 48 para horizon 12 y 24
# ============================================================

CONFIG = {
    "n_nodes": 6,
    "in_channels": 14,
    "hidden": 64,
    "seq_len": SEQ_LEN,
    "horizon": HORIZON,
    "target_idx": 6,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "epochs": 100,
    "batch_size": 32,
    "patience": 15,
    "seed": 42,
}

MODEL_FILE = f"best_model_a3tgcn_h{HORIZON}.pt"
HISTORY_FILE = f"training_history_a3tgcn_h{HORIZON}.csv"


def set_seed(seed):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class STGNNDataset(Dataset):
    def __init__(self, data, mask, seq_len, horizon, target_idx):
        self.data = data; self.mask = mask
        self.seq_len = seq_len; self.horizon = horizon
        self.target_idx = target_idx
        self.n_samples = data.shape[0] - seq_len - horizon + 1
    def __len__(self): return self.n_samples
    def __getitem__(self, i):
        X = self.data[i:i + self.seq_len]
        y = self.data[i + self.seq_len:i + self.seq_len + self.horizon, :, self.target_idx]
        m = self.mask[i + self.seq_len:i + self.seq_len + self.horizon, :, self.target_idx]
        return (torch.tensor(X, dtype=torch.float32),
                torch.tensor(y, dtype=torch.float32),
                torch.tensor(m, dtype=torch.float32))


def batch_edge_index(edge_index, batch_size, n_nodes):
    return torch.cat([edge_index + i * n_nodes for i in range(batch_size)], dim=1)

def batch_edge_weight(edge_weight, batch_size):
    return edge_weight.repeat(batch_size)


class A3TGCNForecaster(nn.Module):
    def __init__(self, in_channels, hidden, horizon, periods):
        super().__init__()
        self.tgnn = A3TGCN(in_channels=in_channels, out_channels=hidden, periods=periods)
        self.head = nn.Linear(hidden, horizon)
    def forward(self, x, edge_index, edge_weight):
        h = self.tgnn(x, edge_index, edge_weight)
        return self.head(h)


def masked_mae_loss(pred, target, mask):
    diff = torch.abs(pred - target) * mask
    n_valid = mask.sum()
    if n_valid == 0:
        return torch.tensor(0.0, device=pred.device, requires_grad=True)
    return diff.sum() / n_valid


def train_epoch(model, loader, optimizer, scaler, edge_index_base,
                edge_weight_base, n_nodes):
    model.train()
    total_loss = 0.0
    n_batches = 0

    for X, y, m in loader:
        bs = X.size(0)
        X, y, m = X.to(DEVICE), y.to(DEVICE), m.to(DEVICE)

        ei = batch_edge_index(edge_index_base, bs, n_nodes).to(DEVICE)
        ew = batch_edge_weight(edge_weight_base, bs).to(DEVICE)

        optimizer.zero_grad()
        with autocast():
            x_in = X.permute(0, 2, 3, 1)
            x_in = x_in.reshape(bs * n_nodes, -1, X.size(1))
            out = model(x_in, ei, ew)
            out = out.reshape(bs, n_nodes, -1).permute(0, 2, 1)
            loss = masked_mae_loss(out, y, m)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / n_batches


@torch.no_grad()
def evaluate(model, loader, edge_index_base, edge_weight_base, n_nodes,
             scalers, target_idx, horizon):
    model.eval()
    all_pred, all_target, all_mask = [], [], []

    for X, y, m in loader:
        bs = X.size(0)
        X = X.to(DEVICE)
        ei = batch_edge_index(edge_index_base, bs, n_nodes).to(DEVICE)
        ew = batch_edge_weight(edge_weight_base, bs).to(DEVICE)

        with autocast():
            x_in = X.permute(0, 2, 3, 1).reshape(bs * n_nodes, -1, X.size(1))
            out = model(x_in, ei, ew)
            out = out.reshape(bs, n_nodes, -1).permute(0, 2, 1)

        all_pred.append(out.cpu())
        all_target.append(y)
        all_mask.append(m)

    pred = torch.cat(all_pred)
    target = torch.cat(all_target)
    mask = torch.cat(all_mask)

    loss_norm = masked_mae_loss(pred, target, mask).item()

    sc = scalers[target_idx]
    pred_np = pred.numpy()
    target_np = target.numpy()
    mask_np = mask.numpy()

    p = sc.inverse_transform(pred_np.flatten().reshape(-1, 1)).flatten()
    t = sc.inverse_transform(target_np.flatten().reshape(-1, 1)).flatten()
    m_flat = mask_np.flatten().astype(bool)

    mae = np.mean(np.abs(t[m_flat] - p[m_flat]))
    rmse = np.sqrt(np.mean((t[m_flat] - p[m_flat]) ** 2))
    ss_res = np.sum((t[m_flat] - p[m_flat]) ** 2)
    ss_tot = np.sum((t[m_flat] - t[m_flat].mean()) ** 2)
    r2 = 1 - ss_res / ss_tot

    metrics_per_step = []
    for h in range(horizon):
        ph = sc.inverse_transform(pred_np[:, h, :].flatten().reshape(-1, 1)).flatten()
        th = sc.inverse_transform(target_np[:, h, :].flatten().reshape(-1, 1)).flatten()
        mh = mask_np[:, h, :].flatten().astype(bool)
        mae_h = np.mean(np.abs(th[mh] - ph[mh]))
        rmse_h = np.sqrt(np.mean((th[mh] - ph[mh]) ** 2))
        r2_h = 1 - np.sum((th[mh] - ph[mh]) ** 2) / np.sum((th[mh] - th[mh].mean()) ** 2)
        metrics_per_step.append({"step": h + 1, "MAE_C": mae_h, "RMSE_C": rmse_h, "R2": r2_h})

    return {
        "loss_norm": loss_norm, "MAE_C": mae, "RMSE_C": rmse, "R2": r2,
        "per_step": pd.DataFrame(metrics_per_step)
    }


if __name__ == "__main__":
    set_seed(CONFIG["seed"])

    print("=" * 60)
    print(f"A3T-GCN - HORIZONTE {HORIZON}h (seq_len={SEQ_LEN})")
    print("=" * 60)
    print(f"Device: {DEVICE}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")

    print(f"\nArchivos de salida:")
    print(f"  Modelo:    {MODEL_FILE}")
    print(f"  Historial: {HISTORY_FILE}")

    print("\nCargando artefactos...")
    edge_index = torch.load("edge_index.pt")
    edge_weight = torch.load("edge_weight.pt")
    with open("scalers.pkl", "rb") as f:
        scalers = pickle.load(f)

    loaders = {}
    for split in ["train", "val", "test"]:
        data = np.load(f"tensor_{split}.npy")
        mask = np.load(f"mask_{split}.npy")
        ds = STGNNDataset(data, mask, CONFIG["seq_len"], CONFIG["horizon"],
                          CONFIG["target_idx"])
        shuffle = (split == "train")
        loaders[split] = DataLoader(
            ds, batch_size=CONFIG["batch_size"], shuffle=shuffle,
            num_workers=0, pin_memory=True, drop_last=False
        )
        print(f"  {split:5s}: {len(ds):,} ventanas")

    model = A3TGCNForecaster(
        in_channels=CONFIG["in_channels"],
        hidden=CONFIG["hidden"],
        horizon=CONFIG["horizon"],
        periods=CONFIG["seq_len"]
    ).to(DEVICE)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nModelo: A3T-GCN (horizon={HORIZON}, periods={SEQ_LEN})")
    print(f"  Parámetros: {n_params:,}")

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"]
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=7, verbose=True
    )
    scaler = GradScaler()

    print(f"\nEntrenando...")
    print(f"{'Epoch':>5s} {'Train':>10s} {'Val MAE°C':>10s} {'Val RMSE°C':>11s} "
          f"{'Val R²':>8s} {'LR':>10s} {'Tiempo':>8s}")
    print("-" * 70)

    best_val_loss = float("inf")
    epochs_no_improve = 0
    history = []

    for epoch in range(1, CONFIG["epochs"] + 1):
        t0 = time.time()

        train_loss = train_epoch(
            model, loaders["train"], optimizer, scaler,
            edge_index, edge_weight, CONFIG["n_nodes"]
        )

        val_metrics = evaluate(
            model, loaders["val"], edge_index, edge_weight,
            CONFIG["n_nodes"], scalers, CONFIG["target_idx"], CONFIG["horizon"]
        )

        elapsed = time.time() - t0
        lr = optimizer.param_groups[0]["lr"]

        print(f"{epoch:5d} {train_loss:10.4f} {val_metrics['MAE_C']:10.4f} "
              f"{val_metrics['RMSE_C']:11.4f} {val_metrics['R2']:8.4f} "
              f"{lr:10.6f} {elapsed:7.1f}s")

        history.append({
            "epoch": epoch, "train_loss": train_loss,
            **{f"val_{k}": v for k, v in val_metrics.items() if k != "per_step"},
            "lr": lr
        })

        scheduler.step(val_metrics["loss_norm"])

        if val_metrics["loss_norm"] < best_val_loss:
            best_val_loss = val_metrics["loss_norm"]
            epochs_no_improve = 0
            torch.save(model.state_dict(), MODEL_FILE)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= CONFIG["patience"]:
                print(f"\nEarly stopping en epoch {epoch}")
                break

    # Evaluación final
    print("\n" + "=" * 60)
    print(f"EVALUACIÓN FINAL EN TEST (horizonte {HORIZON}h)")
    print("=" * 60)

    model.load_state_dict(torch.load(MODEL_FILE))
    test_metrics = evaluate(
        model, loaders["test"], edge_index, edge_weight,
        CONFIG["n_nodes"], scalers, CONFIG["target_idx"], CONFIG["horizon"]
    )

    print(f"\nMétricas globales:")
    print(f"  MAE:  {test_metrics['MAE_C']:.4f} °C")
    print(f"  RMSE: {test_metrics['RMSE_C']:.4f} °C")
    print(f"  R²:   {test_metrics['R2']:.4f}")

    print(f"\nDegradación por paso del horizonte:")
    print(test_metrics["per_step"].to_string(index=False))

    df_hist = pd.DataFrame(history)
    df_hist.to_csv(HISTORY_FILE, index=False)
    test_metrics["per_step"].to_csv(f"metricas_por_paso_a3tgcn_h{HORIZON}.csv", index=False)

    print(f"\nGuardados: {MODEL_FILE}, {HISTORY_FILE}, metricas_por_paso_a3tgcn_h{HORIZON}.csv")

    if torch.cuda.is_available():
        print(f"Memoria GPU pico: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

A3T-GCN - HORIZONTE 6h (seq_len=24)
Device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU

Archivos de salida:
  Modelo:    best_model_a3tgcn_h6.pt
  Historial: training_history_a3tgcn_h6.csv

Cargando artefactos...
  train: 122,707 ventanas
  val  : 17,491 ventanas
  test : 19,652 ventanas

Modelo: A3T-GCN (horizon=6, periods=24)
  Parámetros: 28,062

Entrenando...
Epoch      Train  Val MAE°C  Val RMSE°C   Val R²         LR   Tiempo
----------------------------------------------------------------------
    1     0.3253     1.1032      1.5039   0.8111   0.001000   416.8s
    2     0.2884     1.1445      1.5498   0.7994   0.001000   419.0s
    3     0.2838     1.1293      1.5328   0.8037   0.001000   415.1s
    4     0.2809     1.1410      1.5324   0.8038   0.001000   435.1s
    5     0.2783     1.0987      1.4926   0.8139   0.001000   469.6s
    6     0.2760     1.1038      1.5034   0.8112   0.001000   463.5s
    7     0.2743     1.0840      1.4810   0.8168   0.001000   493.3s
    8    